# Training MobileNetV3-Small + ADAM

**Proyek BHUMI** — Perbandingan Arsitektur CNN untuk Deteksi Penyakit Daun Jagung

Notebook ini menjalankan eksperimen training **MobileNetV3-Small** dengan optimizer **ADAM** menggunakan skema **transfer learning 2 fase**:
- **Phase 1 — Feature Extraction**: Freeze seluruh base model, train classifier baru
- **Phase 2 — Fine-tuning**: Unfreeze sebagian layer atas, train dengan LR lebih kecil

Dataset: preprocessed corn leaf disease (CLAHE → Resize 224×224 → Normalisasi [0,1] → Augmentasi offline)

## 1. Setup & Imports

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1. SETUP & IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

import os
import json
import time
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, transforms

import matplotlib
matplotlib.use('Agg')  # backend non-interaktif untuk Kaggle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    f1_score,
    accuracy_score
)

from tqdm import tqdm

# Tampilkan info environment
print(f"PyTorch version : {torch.__version__}")
print(f"Torchvision     : {torchvision.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

## 2. Konfigurasi

In [ ]:
# =============================================================================
# 2. KONFIGURASI
# =============================================================================
# Semua hyperparameter dikumpulkan di sini agar mudah diubah.

# --- Identitas Eksperimen ---
ARCHITECTURE = "mobilenetv3_small"
OPTIMIZER_NAME = "adam"
OUTPUT_NAME = f"{ARCHITECTURE}_{OPTIMIZER_NAME}"

# --- Dataset ---
# <<< GANTI SLUG DI BAWAH SESUAI NAMA DATASET ANDA DI KAGGLE >>>
DATASET_SLUG = "preprocessed-corn-disease"
DATA_DIR = f"/kaggle/input/{DATASET_SLUG}"
OUTPUT_DIR = f"/kaggle/working/{OUTPUT_NAME}"

# --- Umum ---
BATCH_SIZE = 32
NUM_CLASSES = 4
RANDOM_SEED = 42
NUM_WORKERS = 2

# --- Transfer Learning ---
PHASE1_EPOCHS = 10       # Phase 1: feature extraction (freeze base)
PHASE2_EPOCHS = 20       # Phase 2: fine-tuning (unfreeze sebagian)
UNFREEZE_LAST_N = 4      # jumlah layer terakhir yang di-unfreeze di Phase 2

# --- Learning Rate (Adam) ---
LR_PHASE1 = 1e-3    # learning rate Phase 1 (feature extraction)
LR_PHASE2 = 1e-4    # learning rate Phase 2 (fine-tuning, lebih kecil)

# --- Scheduler (ReduceLROnPlateau) ---
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3

# --- Early Stopping ---
EARLY_STOPPING_PATIENCE = 5   # berbasis val_loss, aktif di Phase 2

# --- Label Mapping ---
# Mapping dari nama folder dataset ke nama kelas yang readable
FOLDER_TO_LABEL = {
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'Gray Leaf Spot',
    'Corn_(maize)___Common_rust_': 'Common Rust',
    'Corn_(maize)___Northern_Leaf_Blight': 'Northern Leaf Blight',
    'Corn_(maize)___healthy': 'Healthy',
}

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Reproducibility ---
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --- Buat direktori output ---
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n{'='*60}")
print(f"  KONFIGURASI EKSPERIMEN")
print(f"{'='*60}")
print(f"  Arsitektur   : {ARCHITECTURE}")
print(f"  Optimizer    : {OPTIMIZER_NAME.upper()}")
print(f"  Device       : {device}")
print(f"  Batch Size   : {BATCH_SIZE}")
print(f"  Phase 1      : {PHASE1_EPOCHS} epochs, LR={LR_PHASE1}")
print(f"  Phase 2      : {PHASE2_EPOCHS} epochs, LR={LR_PHASE2}")
print(f"  Output       : {OUTPUT_DIR}")
print(f"{'='*60}")

## 3. Dataset & DataLoader

In [ ]:
# =============================================================================
# 3. DATASET & DATALOADER
# =============================================================================

# Transform: hanya ToTensor() karena gambar sudah dinormalisasi [0,1]
# saat preprocessing. ToTensor() otomatis konversi PIL Image uint8 [0,255]
# ke Tensor float32 [0,1]. JANGAN tambah normalisasi ImageNet agar tidak
# double-normalize.
data_transform = transforms.Compose([
    transforms.ToTensor(),
])

# Load dataset dari masing-masing split
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, 'train'), transform=data_transform
)
valid_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, 'valid'), transform=data_transform
)
test_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, 'test'), transform=data_transform
)

# DataLoader
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

# Mapping index -> nama kelas readable
# ImageFolder mengurutkan folder secara alfabet untuk class_to_idx
idx_to_folder = {v: k for k, v in train_dataset.class_to_idx.items()}
idx_to_class = {idx: FOLDER_TO_LABEL[folder] for idx, folder in idx_to_folder.items()}

# Tampilkan info dataset
print(f"\n{'='*60}")
print(f"  INFORMASI DATASET")
print(f"{'='*60}")
print(f"  Path  : {DATA_DIR}")
print(f"  Train : {len(train_dataset):>6} gambar")
print(f"  Valid : {len(valid_dataset):>6} gambar")
print(f"  Test  : {len(test_dataset):>6} gambar")
print(f"\n  Mapping Kelas (urutan alfabet ImageFolder):")
for idx in sorted(idx_to_class.keys()):
    folder = idx_to_folder[idx]
    label = idx_to_class[idx]
    count = sum(1 for _, l in train_dataset.samples if l == idx)
    print(f"    [{idx}] {label:<25} <- {folder} ({count} train)")
print(f"{'='*60}")

## 4. Model Definition

In [ ]:
# =============================================================================
# 4. MODEL DEFINITION — MobileNetV3-Small
# =============================================================================

from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


def create_model():
    """
    Load MobileNetV3-Small pretrained (ImageNet) dan modifikasi classifier head.

    Arsitektur MobileNetV3-Small:
        - features: InvertedResidual blocks (frozen di Phase 1)
        - avgpool: adaptive average pooling
        - classifier: Sequential(Linear(576,1024), Hardswish, Dropout, Linear(1024,1000))

    Modifikasi: classifier[-1] (Linear 1024->1000) -> Linear(in_features, NUM_CLASSES)
    """
    model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)

    # Ganti layer terakhir classifier untuk 4 kelas
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)

    return model


def freeze_base_layers(model):
    """Freeze semua layer features (untuk Phase 1 — feature extraction)."""
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True


def unfreeze_last_n_layers(model, n=UNFREEZE_LAST_N):
    """Unfreeze N children terakhir dari features (Phase 2 — fine-tuning)."""
    children = list(model.features.children())
    for child in children[-n:]:
        for param in child.parameters():
            param.requires_grad = True

    total = len(children)
    print(f"\n  Unfreeze {n} layer terakhir dari features (index {total-n}..{total-1}):")
    for i, child in enumerate(children[-n:]):
        print(f"    [{total-n+i}] {type(child).__name__}")


# Buat model dan pindahkan ke device
model = create_model()
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: MobileNetV3-Small")
print(f"Total parameters: {total_params:,}")
print(f"Device: {device}")

## 5. Training Utilities

In [ ]:
# =============================================================================
# 5. TRAINING UTILITIES
# =============================================================================


def train_one_epoch(model, loader, criterion, optimizer, device):
    """Menjalankan satu epoch training. Return: (avg_loss, accuracy)."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="    Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def validate(model, loader, criterion, device):
    """Menjalankan validasi. Return: (avg_loss, accuracy, f1_macro)."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="    Validasi", leave=False):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    total = len(all_labels)
    avg_loss = running_loss / total
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')

    return avg_loss, accuracy, f1


class EarlyStopping:
    """Early stopping berbasis val_loss. Menghentikan training jika val_loss
    tidak membaik selama `patience` epoch berturut-turut."""

    def __init__(self, patience=EARLY_STOPPING_PATIENCE, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# Loss function (Cross-Entropy) digunakan di kedua phase
criterion = nn.CrossEntropyLoss()
print("\nTraining utilities siap.")

## 6. Phase 1 — Feature Extraction

In [ ]:
# =============================================================================
# 6. PHASE 1 — FEATURE EXTRACTION
# =============================================================================
# Freeze semua layer base model (features), hanya train classifier head baru.

print(f"\n{'='*60}")
print(f"  PHASE 1 — FEATURE EXTRACTION")
print(f"  Arsitektur : {ARCHITECTURE}")
print(f"  Optimizer  : {OPTIMIZER_NAME.upper()}")
print(f"  Epochs     : {PHASE1_EPOCHS}")
print(f"  LR         : {LR_PHASE1}")
print(f"  Strategy   : Freeze base, train classifier saja")
print(f"{'='*60}\n")

# Freeze base layers — hanya classifier yang trainable
freeze_base_layers(model)

# Setup optimizer
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE1
)

# Scheduler: kurangi LR jika val_loss stagnan
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min',
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
    verbose=True
)

# Info parameter trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p = sum(p.numel() for p in model.parameters())
print(f"  Trainable parameters: {trainable:,} / {total_p:,} ({trainable/total_p*100:.1f}%)\n")

# --- Training Loop Phase 1 ---
phase1_history = []
best_val_loss_p1 = float('inf')
phase1_start = time.time()

for epoch in range(1, PHASE1_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1 = validate(model, valid_loader, criterion, device)
    scheduler.step(val_loss)

    # Catat history
    phase1_history.append({
        'epoch': epoch,
        'train_loss': round(train_loss, 4),
        'train_acc': round(train_acc, 4),
        'val_loss': round(val_loss, 4),
        'val_acc': round(val_acc, 4),
        'val_f1': round(val_f1, 4),
    })

    # Simpan checkpoint terbaik berdasarkan val_loss
    marker = ""
    if val_loss < best_val_loss_p1:
        best_val_loss_p1 = val_loss
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_model_phase1.pth'))
        marker = " <- best"

    print(f"  Epoch {epoch:2d}/{PHASE1_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}{marker}")

phase1_time = time.time() - phase1_start
print(f"\n  Phase 1 selesai dalam {phase1_time:.1f} detik ({phase1_time/60:.1f} menit)")
print(f"  Best val_loss Phase 1: {best_val_loss_p1:.4f}")

## 7. Phase 2 — Fine-tuning

In [ ]:
# =============================================================================
# 7. PHASE 2 — FINE-TUNING
# =============================================================================
# Load best Phase 1, unfreeze sebagian layer atas, training lanjutan dengan
# learning rate lebih kecil. Early stopping aktif.

print(f"\n{'='*60}")
print(f"  PHASE 2 — FINE-TUNING")
print(f"  Arsitektur : {ARCHITECTURE}")
print(f"  Optimizer  : {OPTIMIZER_NAME.upper()}")
print(f"  Epochs     : {PHASE2_EPOCHS} (max)")
print(f"  LR         : {LR_PHASE2}")
print(f"  Unfreeze   : {UNFREEZE_LAST_N} layer terakhir")
print(f"  Early Stop : patience={EARLY_STOPPING_PATIENCE}")
print(f"{'='*60}")

# Load bobot terbaik dari Phase 1
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model_phase1.pth')))
print("\n  Loaded best_model_phase1.pth")

# Unfreeze N layer terakhir dari feature extractor
unfreeze_last_n_layers(model, UNFREEZE_LAST_N)

# Setup optimizer Phase 2 (LR lebih kecil)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE2
)

# Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min',
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
    verbose=True
)

# Early stopping
early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE)

# Info parameter trainable setelah unfreeze
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p = sum(p.numel() for p in model.parameters())
print(f"\n  Trainable parameters: {trainable:,} / {total_p:,} ({trainable/total_p*100:.1f}%)\n")

# --- Training Loop Phase 2 ---
phase2_history = []
best_val_loss_p2 = float('inf')
phase2_start = time.time()

for epoch in range(1, PHASE2_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1 = validate(model, valid_loader, criterion, device)
    scheduler.step(val_loss)

    # Catat history
    phase2_history.append({
        'epoch': epoch,
        'train_loss': round(train_loss, 4),
        'train_acc': round(train_acc, 4),
        'val_loss': round(val_loss, 4),
        'val_acc': round(val_acc, 4),
        'val_f1': round(val_f1, 4),
    })

    # Simpan checkpoint terbaik berdasarkan val_loss
    marker = ""
    if val_loss < best_val_loss_p2:
        best_val_loss_p2 = val_loss
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_model_phase2.pth'))
        marker = " <- best"

    print(f"  Epoch {epoch:2d}/{PHASE2_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}{marker}")

    # Cek early stopping
    if early_stopping(val_loss):
        print(f"\n  Early stopping triggered di epoch {epoch}")
        break

phase2_time = time.time() - phase2_start
total_training_time = phase1_time + phase2_time
print(f"\n  Phase 2 selesai dalam {phase2_time:.1f} detik ({phase2_time/60:.1f} menit)")
print(f"  Best val_loss Phase 2: {best_val_loss_p2:.4f}")
print(f"  Total waktu training : {total_training_time:.1f} detik ({total_training_time/60:.1f} menit)")

## 8. Evaluasi Test Set

In [ ]:
# =============================================================================
# 8. EVALUASI TEST SET
# =============================================================================
# Evaluasi model terbaik (Phase 2) pada test set yang belum pernah dilihat.

print(f"\n{'='*60}")
print(f"  EVALUASI TEST SET")
print(f"{'='*60}\n")

# Load model terbaik dari Phase 2
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model_phase2.pth')))
model.eval()
print("  Loaded best_model_phase2.pth")

# Inference pada test set
all_preds = []
all_labels = []
running_loss = 0.0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="  Evaluasi test"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# --- Hitung Metrics ---

# Metrics agregat (macro)
test_loss = running_loss / len(all_labels)
test_acc = accuracy_score(all_labels, all_preds)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    all_labels, all_preds, average='macro'
)

# Metrics per kelas
precision_per, recall_per, f1_per, support_per = precision_recall_fscore_support(
    all_labels, all_preds, average=None
)

# Urutan nama kelas sesuai index ImageFolder
class_names_ordered = [idx_to_class[i] for i in range(NUM_CLASSES)]

# --- Tampilkan Hasil ---

print(f"\n  {chr(9472)*55}")
print(f"  HASIL EVALUASI (MACRO-AVERAGED)")
print(f"  {chr(9472)*55}")
print(f"  Test Accuracy   : {test_acc:.4f}")
print(f"  Test Loss       : {test_loss:.4f}")
print(f"  Precision (M)   : {precision_macro:.4f}")
print(f"  Recall (M)      : {recall_macro:.4f}")
print(f"  F1-Score (M)    : {f1_macro:.4f}")

print(f"\n  {chr(9472)*55}")
print(f"  HASIL PER KELAS")
print(f"  {chr(9472)*55}")
print(f"  {'Kelas':<25} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print(f"  {'-'*25} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
for i, name in enumerate(class_names_ordered):
    print(f"  {name:<25} {precision_per[i]:>10.4f} {recall_per[i]:>10.4f} "
          f"{f1_per[i]:>10.4f} {int(support_per[i]):>10}")
print(f"  {chr(9472)*55}")

# --- Classification Report (sklearn) ---
report = classification_report(
    all_labels, all_preds,
    target_names=class_names_ordered,
    digits=4
)
print(f"\n{report}")

# Simpan classification report
report_path = os.path.join(OUTPUT_DIR, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write(f"Classification Report - {ARCHITECTURE.upper()} + {OPTIMIZER_NAME.upper()}\n")
    f.write(f"{'='*60}\n\n")
    f.write(report)
print(f"  Saved: {report_path}")

# --- Simpan evaluation_results.json ---
per_class_dict = {}
for i, name in enumerate(class_names_ordered):
    per_class_dict[name] = {
        'precision': round(float(precision_per[i]), 4),
        'recall': round(float(recall_per[i]), 4),
        'f1': round(float(f1_per[i]), 4),
        'support': int(support_per[i]),
    }

eval_results = {
    'architecture': ARCHITECTURE,
    'optimizer': OPTIMIZER_NAME,
    'test_accuracy': round(float(test_acc), 4),
    'test_loss': round(float(test_loss), 4),
    'test_precision_macro': round(float(precision_macro), 4),
    'test_recall_macro': round(float(recall_macro), 4),
    'test_f1_macro': round(float(f1_macro), 4),
    'per_class': per_class_dict,
    'training_time_phase1_seconds': round(phase1_time, 2),
    'training_time_phase2_seconds': round(phase2_time, 2),
    'training_time_total_seconds': round(total_training_time, 2),
}

eval_path = os.path.join(OUTPUT_DIR, 'evaluation_results.json')
with open(eval_path, 'w') as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)
print(f"  Saved: {eval_path}")

## 9. Visualisasi

In [ ]:
# =============================================================================
# 9. VISUALISASI
# =============================================================================

# Urutan nama kelas untuk label confusion matrix
class_names_ordered = [idx_to_class[i] for i in range(NUM_CLASSES)]

# --- Confusion Matrix ---

cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# 1. Confusion Matrix — Raw (nilai absolut)
fig1, ax1 = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_ordered,
            yticklabels=class_names_ordered, ax=ax1)
ax1.set_title(f'Confusion Matrix (Raw)\n{ARCHITECTURE.upper()} + {OPTIMIZER_NAME.upper()}',
              fontsize=13, fontweight='bold')
ax1.set_xlabel('Prediksi', fontsize=11)
ax1.set_ylabel('Aktual', fontsize=11)
plt.xticks(rotation=25, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
cm_raw_path = os.path.join(OUTPUT_DIR, 'confusion_matrix_raw.png')
plt.savefig(cm_raw_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {cm_raw_path}")
plt.close(fig1)

# 2. Confusion Matrix — Normalized (persentase per baris/true label)
fig2, ax2 = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=class_names_ordered,
            yticklabels=class_names_ordered, ax=ax2)
ax2.set_title(f'Confusion Matrix (Normalized)\n{ARCHITECTURE.upper()} + {OPTIMIZER_NAME.upper()}',
              fontsize=13, fontweight='bold')
ax2.set_xlabel('Prediksi', fontsize=11)
ax2.set_ylabel('Aktual', fontsize=11)
plt.xticks(rotation=25, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
cm_norm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix_normalized.png')
plt.savefig(cm_norm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {cm_norm_path}")
plt.close(fig2)

In [ ]:
# --- Training Curves ---

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Phase 1 — Loss
p1_epochs = [h['epoch'] for h in phase1_history]
axes[0, 0].plot(p1_epochs, [h['train_loss'] for h in phase1_history],
                'b-o', markersize=4, label='Train Loss')
axes[0, 0].plot(p1_epochs, [h['val_loss'] for h in phase1_history],
                'r-o', markersize=4, label='Val Loss')
axes[0, 0].set_title('Phase 1 — Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Phase 1 — Accuracy
axes[0, 1].plot(p1_epochs, [h['train_acc'] for h in phase1_history],
                'b-o', markersize=4, label='Train Acc')
axes[0, 1].plot(p1_epochs, [h['val_acc'] for h in phase1_history],
                'r-o', markersize=4, label='Val Acc')
axes[0, 1].set_title('Phase 1 — Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Phase 2 — Loss
p2_epochs = [h['epoch'] for h in phase2_history]
axes[1, 0].plot(p2_epochs, [h['train_loss'] for h in phase2_history],
                'b-s', markersize=4, label='Train Loss')
axes[1, 0].plot(p2_epochs, [h['val_loss'] for h in phase2_history],
                'r-s', markersize=4, label='Val Loss')
axes[1, 0].set_title('Phase 2 — Loss', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Phase 2 — Accuracy
axes[1, 1].plot(p2_epochs, [h['train_acc'] for h in phase2_history],
                'b-s', markersize=4, label='Train Acc')
axes[1, 1].plot(p2_epochs, [h['val_acc'] for h in phase2_history],
                'r-s', markersize=4, label='Val Acc')
axes[1, 1].set_title('Phase 2 — Accuracy', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle(
    f'Training Curves — {ARCHITECTURE.upper()} + {OPTIMIZER_NAME.upper()}',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
curves_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {curves_path}")
plt.close(fig)

## 10. Simpan Output

In [ ]:
# =============================================================================
# 10. SIMPAN OUTPUT
# =============================================================================

# --- Simpan Training History ---
history = {
    'phase1': phase1_history,
    'phase2': phase2_history,
}
history_path = os.path.join(OUTPUT_DIR, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f"  Saved: {history_path}")

# --- Ringkasan Akhir ---
print(f"\n{'='*60}")
print(f"  RINGKASAN HASIL EKSPERIMEN")
print(f"  {ARCHITECTURE.upper()} + {OPTIMIZER_NAME.upper()}")
print(f"{'='*60}")
print(f"\n  Metrics Test Set:")
print(f"     Accuracy         : {test_acc:.4f}")
print(f"     Loss             : {test_loss:.4f}")
print(f"     Precision (macro): {precision_macro:.4f}")
print(f"     Recall (macro)   : {recall_macro:.4f}")
print(f"     F1-Score (macro) : {f1_macro:.4f}")
print(f"\n  Waktu Training:")
print(f"     Phase 1 : {phase1_time:>8.1f} detik ({phase1_time/60:.1f} menit)")
print(f"     Phase 2 : {phase2_time:>8.1f} detik ({phase2_time/60:.1f} menit)")
print(f"     Total   : {total_training_time:>8.1f} detik ({total_training_time/60:.1f} menit)")
print(f"\n  Output Files ({OUTPUT_DIR}):")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"     {fname} ({size_kb:.1f} KB)")
print(f"\n{'='*60}")
print(f"  Eksperimen {ARCHITECTURE.upper()} + {OPTIMIZER_NAME.upper()} selesai!")
print(f"{'='*60}")